In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import numpy as np
import time
import matplotlib.pyplot as plt
import subprocess
import platform


build_dir = "build"
configType = "Release" #"Debug" Release
#pbindDir = r"c:\Users\CathalMcLoughlin\anaconda3\envs\morpho311\Lib\site-packages\pybind11\share\cmake\pybind11"

# Ensure build directory exists
os.makedirs(build_dir, exist_ok=True)

#automatically identify the pybind11 directory
pbindDir = subprocess.run(
    ["python", "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    capture_output=True, text=True).stdout.strip()

cmakeArgs = [
    'cmake', '-S', ".", '-B', build_dir,
    f"-Dpybind11_DIR={pbindDir}",
    f"-DCMAKE_BUILD_TYPE={configType}"
]

# Portable Fix: If on Mac, point to the Homebrew LLVM compilers
if platform.system() == "Darwin":
    print("on mac")
    # Homebrew usually installs llvm here
    llvm_prefix = "/opt/homebrew/opt/llvm"
    if os.path.exists(llvm_prefix):
        cmakeArgs.append(f"-DCMAKE_C_COMPILER={llvm_prefix}/bin/clang")
        cmakeArgs.append(f"-DCMAKE_CXX_COMPILER={llvm_prefix}/bin/clang++")
        # Help FindOpenMP find the library
        cmakeArgs.append(f"-DOpenMP_ROOT={llvm_prefix}")



# Run CMake Configuration Step
pybind11_path = subprocess.run(
    ["python", "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    capture_output=True, text=True).stdout.strip()


out1 = subprocess.run(cmakeArgs,
 stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print(f"Out1 (CMake Configure): {out1.stdout.decode('utf-8')}")
print(f"Err1 (CMake Configure): {out1.stderr.decode('utf-8')}")

# If configuration fails, exit early
if out1.returncode != 0:
    print("CMake configuration failed. Exiting.")
    exit(1)

# Run CMake Build Step
out2 = subprocess.run(['cmake', '--build', build_dir, '--config', configType], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f"Out2 (CMake Build): {out2.stdout.decode('utf-8')}")
print(f"Err2 (CMake Build): {out2.stderr.decode('utf-8')}")

# If build fails, print error
if out2.returncode != 0:
    print("CMake build failed.")
    assert False

on mac
Out1 (CMake Configure): -- The CXX compiler identification is Clang 21.1.8
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /opt/homebrew/opt/llvm/bin/clang++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /Users/cathal/miniconda3/envs/morphoEnv/bin/python3.13 (found version "3.13.12") found components: Interpreter Development Development.Module Development.Embed
-- Performing Test HAS_FLTO_THIN
-- Performing Test HAS_FLTO_THIN - Success
-- Performing Test HAS_FLTO
-- Performing Test HAS_FLTO - Success
-- Found pybind11: /Users/cathal/miniconda3/envs/morphoEnv/lib/python3.13/site-packages/pybind11/include (found version "3.0.2")
-- Found OpenMP_CXX: -fopenmp=libomp (found version "5.1")
-- Found OpenMP: TRUE (found version "5.1")
-- Configuring done (0.9s)
-- Generating done (0.0s)
-- Build files have been written to: /Users/cathal/Documents/GitHub/pythonCPPtest

In [2]:
moduleDir = os.path.join(build_dir, configType)
if moduleDir not in sys.path:
    sys.path.append(moduleDir)

import cathalTest

print(dir(cathalTest))  

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ompMaxThreads', 'paritySum', 'solveNeuralSystem', 'solveNeuralSystemSingleThread', 'testFunc']


In [3]:
cathalTest.testFunc(np.array([1,1,2],dtype=int),np.array([0.0,1.1,2.1],dtype=np.float64))

In [4]:
cathalTest.paritySum(20)

10

In [6]:
cathalTest.ompMaxThreads()

10

In [4]:
myDict = {"neuronId": np.arange(3),
          "voltages":np.random.random(5),}

cathalTest.solveNeuralSystem(myDict,2,0.1)

TypeError: solveNeuralSystem(): incompatible function arguments. The following argument types are supported:
    1. (myDict: dict, recordingInfo: dict, timeStep: typing.SupportsFloat, nIterations: typing.SupportsInt, samplingInterval: typing.SupportsInt) -> object

Invoked with: {'neuronId': array([0, 1, 2]), 'voltages': array([0.45852292, 0.19764658, 0.21065638, 0.71810505, 0.94416906])}, 2, 0.1